In [2]:
import pandas as pd
import json

In [3]:
business = pd.read_json("yelp_academic_dataset_business.json", lines = True)

business = business[
    ['business_id', 'name', 'city', 'state', 'stars', 'review_count', 'categories','attributes']
]

business = business.dropna(subset=['categories'])

business = business[business['categories'].str.contains('Restaurants', na=False)]

business['categories'] = business['categories'].str.split(', ')


In [4]:
business['price'] = business['attributes'].apply(
    lambda x: x.get('RestaurantsPriceRange2') if isinstance(x,dict) else None
)

business['price'] = pd.to_numeric(business['price'], errors = 'coerce')

def price_to_tier(p):
    if p == 1:
        return '$'
    elif p == 2:
        return '$$'
    elif p == 3:
        return '$$$'
    elif p == 4:
        return '$$$$'
    else:
        return None
business['tier'] = business['price'].apply(price_to_tier)
business = business.drop(columns=['price'], errors='ignore')
business = business[business['tier'].notna()]

In [5]:
def extract_cuisine(category_list):
    category_list = [c.lower() for c in category_list]

    def contains_any(keywords):
        return any(any(k in c for k in keywords) for c in category_list)
    if contains_any(['american (traditional)', 'american (new)', 'burgers']):
        return 'American'
    if contains_any(['indian']):
        return 'Indian'
    if contains_any(['chinese', 'szechuan', 'cantonese']):
        return 'Chinese'
    if contains_any(['mexican']):
        return 'Mexican'
    if contains_any(['italian']):
        return 'Italian'
    if contains_any(['japanese', 'sushi', 'ramen']):
        return 'Japanese'
    if contains_any(['thai']):
        return 'Thai'
    if contains_any(['korean']):
        return 'Korean'
    if contains_any(['greek']):
        return 'Greek'
    if contains_any(['french']):
        return 'French'

    return None
business['cuisine'] = business['categories'].apply(extract_cuisine)

business_filtered = business[business['cuisine'].notna()]

print(business_filtered['cuisine'].value_counts())

cuisine
American    14219
Italian      3488
Mexican      3343
Chinese      2658
Japanese     1629
Indian        648
Thai          507
Greek         492
French        240
Korean        226
Name: count, dtype: int64


In [6]:
chains_to_group = [
    "Taco Bell", "Pizza Hut", "Chipotle Mexican Grill", "Panda Express",
    "Jack in the Box", "QDOBA Mexican Eats", "Moe's Southwest Grill",
    "Olive Garden Italian Restaurant", "Domino's Pizza", "Jet's Pizza",
    "Carrabba's Italian Grill", "Boston Pizza", "Tijuana Flats",
    "Zoes Kitchen", "Noodles & Company", "Fazoli's",
    "Romano's Macaroni Grill", "P.F. Chang's",
    "Papa John's Pizza", "Pei Wei"
]

chains_df = business_filtered[business_filtered['name'].isin(chains_to_group)]
non_chains_df = business_filtered[~business_filtered['name'].isin(chains_to_group)]
chains_grouped = chains_df.drop_duplicates(subset=['name','state'])

business_filtered = pd.concat([non_chains_df, chains_grouped])

In [7]:
business_ids = set(business_filtered['business_id'])

filtered_reviews = []

with open("yelp_academic_dataset_review.json", 'r') as f:
    for i, line in enumerate(f):
        review = json.loads(line)
        review_date = review['date'][:10]
        
        if review_date >= "2021-01-01" and review['business_id'] in business_ids:
                filtered_reviews.append({
                    'business_id': review['business_id'],
                    'text': review['text'],
                    'stars': review['stars'],
                    'date': review['date'],
                    'user_id': review['user_id']
                })
        
        if i % 500000 == 0:
            print(f"Processed {i} lines, {len(filtered_reviews)} reviews collected...")

if len(filtered_reviews) == 0:
     print("No reviews found")

Processed 0 lines, 0 reviews collected...
Processed 500000 lines, 3590 reviews collected...
Processed 1000000 lines, 26329 reviews collected...
Processed 1500000 lines, 51545 reviews collected...
Processed 2000000 lines, 60631 reviews collected...
Processed 2500000 lines, 76334 reviews collected...
Processed 3000000 lines, 98740 reviews collected...
Processed 3500000 lines, 121452 reviews collected...
Processed 4000000 lines, 122593 reviews collected...
Processed 4500000 lines, 147299 reviews collected...
Processed 5000000 lines, 174327 reviews collected...
Processed 5500000 lines, 188029 reviews collected...
Processed 6000000 lines, 199867 reviews collected...
Processed 6500000 lines, 225419 reviews collected...


In [8]:
reviews = pd.DataFrame(filtered_reviews)
reviews['date'] = pd.to_datetime(reviews['date'])
reviews['text'] = reviews['text'].str.lower()
reviews = reviews[reviews['text'].str.len() > 20]

In [9]:
reviews.to_csv("clean_reviews.csv", index=False)

In [10]:
final_df = reviews.merge(business_filtered, on='business_id')
final_df = final_df[['business_id',
        'name',
        'cuisine',
        'city',
        'state',
        'stars_y',
        'review_count',
        'categories',
        'tier',
        'user_id',
        'stars_x',
        'text',
        'date']]
final_df = final_df.drop(columns=['attributes'], errors='ignore')
print(final_df.shape)
print(final_df['cuisine'].value_counts())

(251064, 13)
cuisine
American    142816
Mexican      28866
Italian      27254
Japanese     19193
Chinese      14321
Thai          5369
Indian        4901
Greek         3639
Korean        2386
French        2319
Name: count, dtype: int64


In [11]:
final_df['text'] = final_df['text'].str.lower()

business_filtered = business_filtered.drop(columns=['attributes', 'price'], errors='ignore')

final_df = final_df[final_df['text'].str.len() > 20]

final_df = final_df.rename(columns={
    'stars_x': 'individual_stars',
    'stars_y': 'total_business_stars'
})

In [12]:
final_df.to_csv("final_yelp_dataset.csv", index=False)
final_df.to_json("final_yelp_dataset.json", index = False)

In [13]:
user_ids = set(final_df['user_id'])
filtered_users = []

with open("yelp_academic_dataset_user.json", 'r') as f:
    for i, line in enumerate(f):
        user = json.loads(line)
        
        if user['user_id'] in user_ids:
            filtered_users.append({
                'user_id': user['user_id'],
                'review_count': user['review_count'],
                'elite': user['elite'],
                'average_stars': user['average_stars']
            })
        
        if i % 100000 == 0:
            print(f"Processed {i} users...")

Processed 0 users...
Processed 100000 users...
Processed 200000 users...
Processed 300000 users...
Processed 400000 users...
Processed 500000 users...
Processed 600000 users...
Processed 700000 users...
Processed 800000 users...
Processed 900000 users...
Processed 1000000 users...
Processed 1100000 users...
Processed 1200000 users...
Processed 1300000 users...
Processed 1400000 users...
Processed 1500000 users...
Processed 1600000 users...
Processed 1700000 users...
Processed 1800000 users...
Processed 1900000 users...


In [14]:
users = pd.DataFrame(filtered_users)

users['is_elite'] = users['elite'].apply(lambda x: 0 if x == 'None' else 1)

users = users.drop(columns=['elite']) 


In [15]:
users.to_csv("clean_users.csv", index=False)

In [16]:
final_df.groupby('cuisine')['business_id'].nunique()

cuisine
American    8650
Chinese     1600
French        99
Greek        256
Indian       297
Italian     1858
Japanese     970
Korean       122
Mexican     1692
Thai         298
Name: business_id, dtype: int64

In [17]:
restaurants_per_cuisine = final_df.groupby('cuisine')['business_id'].nunique()
reviews_per_cuisine = final_df['cuisine'].value_counts()
stats = pd.DataFrame({
    'Restaurants': restaurants_per_cuisine,
    'Reviews': reviews_per_cuisine
}).fillna(0).astype(int)

print(stats)

          Restaurants  Reviews
cuisine                       
American         8650   142816
Chinese          1600    14321
French             99     2319
Greek             256     3639
Indian            297     4901
Italian          1858    27254
Japanese          970    19193
Korean            122     2386
Mexican          1692    28866
Thai              298     5369


In [18]:
final_df['business_id'].nunique()

15842